In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


DATA_PATH = "../Data/Processed/"

In [7]:
df_all = pd.read_csv(DATA_PATH + "result_without_user_no_promotion.csv")

In [8]:
df_all['reqDate'] = pd.to_datetime(df_all['reqDate'])

# Chỉ giữ lại phần Ngày (Bỏ Giờ Phút Giây)
df_all['reqDate'] = pd.to_datetime(df_all['reqDate'].dt.date)
df_all['campaignID'] = df_all['campaignID'].astype(str)

# Lọc các giao dịch thành công
df_all = df_all[df_all['transStatus'] == 1] 

### Clean Invalid sub-cat

In [9]:
# 1. Giao dịch nội bộ (Luân chuyển tiền hệ thống / Nhân sự)
INTERNAL_SUB_CATS = [
    'Internal_Service_Education',
    'Internal_Service_Facility',
    'Internal_Service_Internal_Use',
]
# 2. Giao dịch lỗi / Chưa xác định (làm nhiễu data)
#    - Toàn bộ mã có đuôi 'Unknown_Type'
#    - 'Unknown_Group'
mask_unknown_type  = df_all['report_sub_cat'].str.endswith('Unknown_Type', na=False)
mask_unknown_group = df_all['report_sub_cat'] == 'Unknown_Group'
# 3. Quyên góp từ thiện (MDR = 0%, không tạo doanh thu)
CHARITY_SUB_CATS = [
    'Social_Service_Contribution',
]
# Tổng hợp mask loại bỏ
mask_to_remove = (
    df_all['report_sub_cat'].isin(INTERNAL_SUB_CATS)
    | mask_unknown_type
    | mask_unknown_group
    | df_all['report_sub_cat'].isin(CHARITY_SUB_CATS)
)

In [10]:
# --- Thống kê trước khi lọc ---
print(f"Số dòng trước khi lọc : {len(df_all):,}")
print(f"Số dòng bị loại bỏ    : {mask_to_remove.sum():,}")
print("\nChi tiết các sub_cat bị loại:")
print(df_all[mask_to_remove]['report_sub_cat'].value_counts().to_string())
# --- Áp dụng lọc ---
df_all = df_all[~mask_to_remove].copy()
print(f"\nSố dòng sau khi lọc  : {len(df_all):,}")

Số dòng trước khi lọc : 271,752
Số dòng bị loại bỏ    : 4,632

Chi tiết các sub_cat bị loại:
report_sub_cat
Unknown_Type                     3841
Internal_Service_Education        468
Internal_Service_Internal_Use     217
Social_Service_Contribution        87
Internal_Service_Facility          19

Số dòng sau khi lọc  : 267,120


## Độ phủ dịch vụ (Category Breadth)

In [11]:
# Đếm số danh mục duy nhất mỗi user sử dụng trong từng Campaign
user_cat_breadth = df_all.groupby(['campaignID', 'userID'])['report_cat'].nunique().reset_index()
user_cat_breadth.rename(columns={'report_cat': 'unique_categories'}, inplace=True)

# Tính trung bình độ phủ dịch vụ của tập user theo từng Campaign
campaign_cat_breadth = user_cat_breadth.groupby('campaignID')['unique_categories'].mean().reset_index()
campaign_cat_breadth.rename(columns={'unique_categories': 'avg_categories_per_user'}, inplace=True)

# Sắp xếp để xem Campaign nào mang lại user có độ phủ dịch vụ cao nhất
campaign_cat_breadth = campaign_cat_breadth.sort_values(by='avg_categories_per_user', ascending=False)

fig_breadth = px.bar(
    campaign_cat_breadth.head(20), 
    x='campaignID', 
    y='avg_categories_per_user',
    title='Trung bình số lượng dịch vụ (Category Breadth) theo từng Campaign',
    labels={'campaignID': 'Campaign', 'avg_categories_per_user': 'Trung bình số dịch vụ/User'},
    color='avg_categories_per_user',
    color_continuous_scale='Blues'
)

fig_breadth.update_layout(xaxis_tickangle=-45)
fig_breadth.show()

- Trục Y hiển thị giá trị trung bình. Gần như tất cả các chiến dịch Marketing trả phí (10488, 10489, 9954...) đều có một cột phẳng lì, dừng lại ở mức chính xác hoặc xấp xỉ 1.0 dịch vụ/user. Chỉ có cột số 0 (Organic/User tự nhiên) là nhô cao lên mức 1.5.

- Các chiến dịch Marketing hoàn toàn thất bại trong việc cross-sell. 100% người dùng mang về từ quảng cáo là 'thợ săn khuyến mãi' dùng đúng 1 lần rồi bỏ app."

$\rightarrow$ Nhưng liệu chất lượng người dùng có thực sự tệ đến thế? Nghi ngờ có một "lỗi đứt gãy ghi nhận" (Attribution Drop-off): Hệ thống chỉ gắn mã Campaign cho giao dịch click quảng cáo đầu tiên, còn các giao dịch tự phát sau đó bị bỏ trống hoặc tính về 0. Chúng ta buộc phải dùng kỹ thuật để nối lại vòng đời của họ.

## Sử dụng First-Touch Attribution

In [12]:
df_real_camp = df_all[df_all['campaignID'] != 0].dropna(subset=['campaignID'])
first_touch = df_real_camp.groupby('userID')['campaignID'].first().reset_index()
first_touch.rename(columns={'campaignID': 'acquisition_campaign'}, inplace=True)

# Gộp Campaign đầu tiên này ngược lại vào bảng giao dịch chính
df_fixed = df_all.merge(first_touch, on='userID', how='left')

# Với những user hoàn toàn tự nhiên (không qua campaign nào), ta điền mặc định là 'Organic' hoặc '0'
df_fixed['acquisition_campaign'] = df_fixed['acquisition_campaign'].fillna('0')

# Đếm số danh mục (report_cat) duy nhất mỗi user sử dụng dựa trên Campaign đầu tiên của họ
user_cat_counts_fixed = df_fixed.groupby(['acquisition_campaign', 'userID'])['report_cat'].nunique().reset_index(name='unique_categories')

# Tính số lượng user cho từng mức độ phủ (1, 2, 3...) trong mỗi Campaign
dist_data_fixed = user_cat_counts_fixed.groupby(['acquisition_campaign', 'unique_categories']).size().reset_index(name='user_count')

# Tạo cột nhóm độ phủ để biểu đồ gọn gàng hơn
dist_data_fixed['category_group'] = dist_data_fixed['unique_categories'].apply(
    lambda x: '4+ danh mục' if x >= 4 else f'{x} danh mục'
)

# Đếm tổng lượng user của từng Campaign để lọc ra Top 10 lớn nhất
campaign_volume = dist_data_fixed.groupby('acquisition_campaign')['user_count'].sum().reset_index(name='total_users')
top_10_campaigns = campaign_volume.sort_values('total_users', ascending=False).head(10)['acquisition_campaign'].tolist()

# Ép các Campaign nhỏ lẻ bên ngoài Top 10 thành nhóm "Others"
dist_data_fixed['campaign_grouped'] = dist_data_fixed['acquisition_campaign'].apply(
    lambda x: str(x) if x in top_10_campaigns else 'Others'
)

# Tính toán lại tổng số lượng user sau khi đã gom nhóm "Others"
final_dist_grouped = dist_data_fixed.groupby(['campaign_grouped', 'category_group'])['user_count'].sum().reset_index()

# Tính tỷ lệ % user cho từng nhóm mức độ phủ trên tổng số user của Campaign đó
final_dist_grouped['pct_users'] = final_dist_grouped.groupby('campaign_grouped')['user_count'].transform(lambda x: (x / x.sum()) * 100)

# Thiết lập thứ tự sắp xếp trục X (Top 10 từ lớn đến nhỏ, Others nằm cuối cùng)
# Chuyển list sang dạng string để map chính xác với trục X của Plotly
order_x = [str(c) for c in top_10_campaigns] + ['Others']

# Vẽ biểu đồ Cột chồng (Stacked Bar Chart)
fig_clean = px.bar(
    final_dist_grouped, 
    x='campaign_grouped', 
    y='pct_users', 
    color='category_group', 
    title='Phân phối Độ phủ dịch vụ - Top 10 Campaign Lớn Nhất (Mô hình First-Touch Attribution)',
    labels={
        'campaign_grouped': 'Chiến dịch (Top 10 + Others)', 
        'pct_users': 'Tỷ lệ NPU (%)', 
        'category_group': 'Mức độ phủ'
    },
    text_auto='.1f', # Hiển thị số % trên thân cột
    category_orders={
        "campaign_grouped": order_x,
        "category_group": ["1 danh mục", "2 danh mục", "3 danh mục", "4+ danh mục"]
    },
    color_discrete_sequence=px.colors.sequential.Teal_r # Dải màu xanh Teal dễ nhìn
)

# Tối ưu giao diện biểu đồ
fig_clean.update_layout(
    xaxis_tickangle=-45, 
    yaxis_title="Tỷ lệ NPU (%)",
    legend_title="Mức độ phủ",
    barmode='stack'
)

fig_clean.show()

- Khi gán mã chiến dịch đầu tiên cho toàn bộ lịch sử phía sau của user, sự thật được phơi bày. Khối màu xanh đậm (1 danh mục) co hẹp lại, nhường chỗ cho các dải màu mở rộng. Đặc biệt là Campaign 8248 trở thành "nhà vô địch" khi có tới 92.6% user mở rộng sang dịch vụ thứ 2, thứ 3.

- Ý nghĩa: Mô hình này chứng minh: Chiến dịch gieo mầm ban đầu có công mang về một tập user có tiềm năng gắn kết rất cao trong tương lai.

In [13]:
df_success = df_all.copy()

df_success['campaign_clean'] = df_success['campaignID'].replace(0, np.nan)

# Dùng ffill() theo từng userID: Giao dịch tự nhiên phía sau sẽ lấy mã Campaign thực tế gần nhất trước đó.
# Nếu gặp một Campaign mới khác 0, nó sẽ tự động cập nhật theo Campaign mới đó.
df_success['last_touch_campaign'] = df_success.groupby('userID')['campaign_clean'].ffill()

# Những user hoàn toàn tự nhiên ngay từ đầu (không có camp nào trước đó), điền mặc định là '0' (Organic)
df_success['last_touch_campaign'] = df_success['last_touch_campaign'].fillna('0')


# ==========================================
# BƯỚC 3: TẠO BẢNG ĐẾM ĐỘ PHỦ DỊCH VỤ (ĐÃ SỬA)
# ==========================================
# Đếm số danh mục duy nhất mỗi user sử dụng dựa trên Campaign gần nhất gánh vác giao dịch đó
user_cat_counts_fixed = df_success.groupby(['last_touch_campaign', 'userID'])['report_cat'].nunique().reset_index(name='unique_categories')


# ==========================================
# BƯỚC 4: GOM NHÓM ĐỘ PHỦ VÀ TÌM TOP 10 CAMPAIGN
# ==========================================
# Tính số lượng user cho từng mức độ phủ (1, 2, 3...) trong mỗi Campaign Last-Touch
dist_data_fixed = user_cat_counts_fixed.groupby(['last_touch_campaign', 'unique_categories']).size().reset_index(name='user_count')

# Tạo cột nhóm độ phủ để biểu đồ gọn gàng hơn
dist_data_fixed['category_group'] = dist_data_fixed['unique_categories'].apply(
    lambda x: '4+ danh mục' if x >= 4 else f'{x} danh mục'
)

# Đếm tổng lượng user của từng Campaign để lọc ra Top 10 lớn nhất
campaign_volume = dist_data_fixed.groupby('last_touch_campaign')['user_count'].sum().reset_index(name='total_users')

# Lọc danh sách Top 10 (loại trừ mã '0' ra nếu có để tìm đúng 10 Campaign Marketing thực tế)
top_10_campaigns = campaign_volume[campaign_volume['last_touch_campaign'] != '0'].sort_values('total_users', ascending=False).head(10)['last_touch_campaign'].tolist()

# Ép các Campaign nhỏ lẻ bên ngoài Top 10 thành nhóm "Others", và giữ riêng nhóm '0' thành "Organic (0)"
dist_data_fixed['campaign_grouped'] = dist_data_fixed['last_touch_campaign'].apply(
    lambda x: 'Organic (0)' if x == '0' else (str(int(float(x))) if x in top_10_campaigns else 'Others')
)


# ==========================================
# BƯỚC 5: TÍNH TỶ LỆ PHẦN TRĂM VÀ VẼ BIỂU ĐỒ
# ==========================================
# Tính toán lại tổng số lượng user sau khi đã phân loại vào các nhóm mới
final_dist_grouped = dist_data_fixed.groupby(['campaign_grouped', 'category_group'])['user_count'].sum().reset_index()

# Tính tỷ lệ % user cho từng nhóm mức độ phủ trên tổng số user của Campaign đó
final_dist_grouped['pct_users'] = final_dist_grouped.groupby('campaign_grouped')['user_count'].transform(lambda x: (x / x.sum()) * 100)

# Thiết lập thứ tự sắp xếp trục X: Top 10 Campaign -> cột Organic (0) -> cột Others cuối cùng
order_x = [str(int(float(c))) for c in top_10_campaigns] + ['Organic (0)', 'Others']

# Vẽ biểu đồ Cột chồng (Stacked Bar Chart) theo Last-Touch
fig_clean = px.bar(
    final_dist_grouped, 
    x='campaign_grouped', 
    y='pct_users', 
    color='category_group', 
    title='Phân phối Độ phủ dịch vụ - Top 10 Campaign Lớn Nhất (Mô hình Last-Touch Attribution)',
    labels={
        'campaign_grouped': 'Chiến dịch (Top 10 + Groups)', 
        'pct_users': 'Tỷ lệ NPU (%)', 
        'category_group': 'Mức độ phủ'
    },
    text_auto='.1f', 
    category_orders={
        "campaign_grouped": order_x,
        "category_group": ["1 danh mục", "2 danh mục", "3 danh mục", "4+ danh mục"]
    },
    color_discrete_sequence=px.colors.sequential.Teal_r 
)

# Tối ưu giao diện biểu đồ
fig_clean.update_layout(
    xaxis_tickangle=-45, 
    yaxis_title="Tỷ lệ NPU (%)",
    legend_title="Mức độ phủ",
    barmode='stack'
)

fig_clean.show()

- Insight từ hình: Con số này không sai, nó phản ánh một hành vi thực tế: User cực kỳ thực dụng. Trong phạm vi một chiến dịch cụ thể, họ chỉ lột tả đúng 1 nhu cầu. Họ không tự nhiên dùng chéo dịch vụ nếu Campaign đó không có ưu đãi. Để chuyển sang danh mục thứ 2, họ sẽ chủ động thoát ra và kích hoạt một Campaign mới.

- Việc Last-Touch chặt nhỏ lịch sử user làm mất đi bức tranh tổng quan về "Độ phủ vòng đời". Vì vậy, để đánh giá chất lượng tệp khách hàng mà Marketing mang lại, Biểu đồ 2 (First-Touch) mang giá trị chiến lược cao hơn. Nó dẫn dắt chúng ta đến bước cuối cùng: Bóc tách xem tệp user "săn nhiều campaign" này thực chất là ai?

In [14]:
# 1. Đọc dữ liệu
df = pd.read_csv(DATA_PATH + 'result_clean.csv')

# 2. Lọc bỏ các giao dịch tự nhiên (campaignID == 0) để đếm số Campaign thực tế
df_real_campaigns = df[df['campaignID'] != 0].dropna(subset=['campaignID'])

# Đếm số Campaign mà mỗi user đã săn
user_campaigns = df_real_campaigns.groupby('userID')['campaignID'].nunique().reset_index(name='num_campaigns')

# Đếm số lượng dịch vụ (Category Breadth) mà mỗi user đã dùng (tính cả tự nhiên)
user_categories = df.groupby('userID')['report_cat'].nunique().reset_index(name='unique_categories')

# Gộp 2 bảng
user_metrics = pd.merge(user_categories, user_campaigns, on='userID', how='left')
user_metrics['num_campaigns'] = user_metrics['num_campaigns'].fillna(0)

# 3. Phân mảnh người dùng (User Segmentation)
def segment_user(num):
    if num <= 1:
        return '1. Organic/Loyal (0-1 Campaign)'
    elif 2 <= num <= 5:
        return '2. Deal Seekers (2-5 Campaigns)'
    else:
        return '3. Hardcore Promo Hunters (>5 Campaigns)'

user_metrics['Segment'] = user_metrics['num_campaigns'].apply(segment_user)

# 4. Phân tích: Tính trung bình số dịch vụ (Category Breadth) theo từng Segment
segment_analysis = user_metrics.groupby('Segment').agg(
    Total_Users=('userID', 'count'),
    Avg_Category_Breadth=('unique_categories', 'mean')
).reset_index()

# Tính % User cho mỗi tập
segment_analysis['User_Percentage'] = (segment_analysis['Total_Users'] / segment_analysis['Total_Users'].sum()) * 100
segment_analysis

,Segment,Total_Users,Avg_Category_Breadth,User_Percentage
0,1. Organic/Loyal (0-1 Campaign),15609,1.262092,46.395981
1,2. Deal Seekers (2-5 Campaigns),14570,1.956211,43.307672
2,3. Hardcore Promo Hunters (>5 Campaigns),3464,3.351905,10.296347


- Chia người dùng thành 3 nhóm dựa trên số lượng Campaign họ đã tham gia. Trục Y thể hiện trung bình số dịch vụ thực tế họ đã chạm tới trên app.

- Biểu đồ này lật ngược hoàn toàn mọi định kiến định tính thông thường:

    - Nhóm Organic/Loyal (0-1 Campaign) chiếm tới 46.4% tập user, tưởng là chất lượng nhất nhưng thực ra lại lười nhất, độ phủ dịch vụ thấp nhất (1.26). Họ chỉ dùng đúng một tính năng cố định rồi thoát.

    - Nhóm Hardcore Promo Hunters (>5 Campaigns) chiếm 10.3%, thường bị coi là "user rác/user ảo", nhưng lại là nhóm tương tác sâu nhất với hệ sinh thái app khi dùng trung bình tới 3.35 dịch vụ khác nhau.

## **Nhận xét**

Khuyến mãi là "Cửa ngõ" mở rộng độ phủ: Người dùng của ứng dụng không tự nhiên mở rộng dịch vụ theo kiểu Organic. Chính các chiến dịch khuyến mãi liên tục là chất xúc tác, là động lực khảo sát buộc họ phải đi xuyên qua các danh mục khác nhau của ứng dụng. Nhóm user càng tương tác với nhiều Campaign thì độ phủ dịch vụ của họ càng rộng (từ 1.26 nhảy vọt lên 3.35).

Chiến lược hành động cho Doanh nghiệp:

- Về phía Marketing: Lấy công thức target và thiết kế luồng của Campaign 7424 (Hình 2) làm chuẩn để nhân bản, vì đây là chiến dịch mang lại tập user có gen "mở rộng dịch vụ" tốt nhất. Đồng thời, khai tử hoặc tái cấu trúc các chiến dịch có tỷ lệ ngõ cụt cao như 8932.

- Về phía Sản phẩm (Product UI/UX): Tận dụng hành vi của nhóm "Thợ săn mã" bằng cách thiết kế các Gamification (nhiệm vụ nhận quà) hoặc Voucher chéo hệ sinh thái (Ví dụ: Thanh toán hóa đơn xong sẽ tặng mã giảm giá Mua sắm). Đây là cách biến những user Organic đang "lười" ở mức 1.26 dịch vụ phải bắt đầu hành trình khám phá các tính năng khác của app.

## Phân tích Đường dẫn hành vi (Behavioral Path & Cross-selling)

In [15]:
# 1. Sắp xếp giao dịch theo thời gian của từng User
df_all = df_all.sort_values(by=['userID', 'reqDate'])

# 2. Đánh số thứ tự giao dịch (Rank) cho mỗi user
df_all['trans_rank'] = df_all.groupby('userID').cumcount() + 1

# 3. Lấy giao dịch ĐẦU TIÊN của mỗi user
first_trans = df_all[df_all['trans_rank'] == 1][['userID', 'campaignID', 'report_cat']]
first_trans.rename(columns={'report_cat': 'first_category', 'campaignID': 'first_campaign'}, inplace=True)

# 4. Lọc ra những user có giao dịch đầu tiên là "Thanh toán hóa đơn"
target_users = first_trans[first_trans['first_category'] == 'Obligation_Payment'].copy()

# 5. Tìm các giao dịch TỪ LẦN THỨ 2 TRỞ ĐI của nhóm user này
subsequent_trans = df_all[(df_all['userID'].isin(target_users['userID'])) & (df_all['trans_rank'] > 1)]

# 6. Xác định những user có cross-sell (phát sinh giao dịch ở danh mục khác hóa đơn)
cross_sell_users = subsequent_trans[subsequent_trans['report_cat'] != 'Obligation_Payment']['userID'].unique()

# 7. Tính tỷ lệ chuyển đổi cho từng Campaign
target_users['is_cross_sell'] = target_users['userID'].apply(lambda x: 1 if x in cross_sell_users else 0)

cross_sell_report = target_users.groupby('first_campaign').agg(
    total_users=('userID', 'count'),
    cross_sell_users=('is_cross_sell', 'sum')
).reset_index()

# Tính % Cross-sell
cross_sell_report['cross_sell_rate (%)'] = (cross_sell_report['cross_sell_users'] / cross_sell_report['total_users']) * 100
cross_sell_report = cross_sell_report.sort_values(by='cross_sell_rate (%)', ascending=False)
print(cross_sell_report)

   first_campaign  total_users  cross_sell_users  cross_sell_rate (%)
1           10129            1                 1                100.0
3           10154            1                 1                100.0
4           10159            1                 1                100.0
12          10526            3                 3                100.0
10          10356            1                 1                100.0
..            ...          ...               ...                  ...
69           9082            5                 0                  0.0
84           9738            1                 0                  0.0
80           9519            1                 0                  0.0
92           9931            1                 0                  0.0
93           9938            2                 0                  0.0

[94 rows x 4 columns]


In [16]:
# Lấy giao dịch lần 1 và lần 2 của mỗi user
trans_1 = df_all[df_all['trans_rank'] == 1][['userID', 'report_cat']]
trans_1.rename(columns={'report_cat': 'First_Step'}, inplace=True)

trans_2 = df_all[df_all['trans_rank'] == 2][['userID', 'report_cat']]
trans_2.rename(columns={'report_cat': 'Second_Step'}, inplace=True)

# Gộp lại để tạo luồng: First_Step -> Second_Step
path_df = trans_1.merge(trans_2, on='userID', how='inner')

heatmap_data = path_df.groupby(['First_Step', 'Second_Step']).size().reset_index(name='Count')
heatmap_pivot = heatmap_data.pivot(index='First_Step', columns='Second_Step', values='Count').fillna(0)

# 2. Vẽ Heatmap
fig_heatmap = px.imshow(
    heatmap_pivot,
    labels=dict(x="Giao dịch lần 2", y="Giao dịch lần 1", color="Số lượng User"),
    x=heatmap_pivot.columns,
    y=heatmap_pivot.index,
    text_auto=True, # Hiển thị số trực tiếp trên ô
    aspect="auto",
    color_continuous_scale='Blues',
    title="Ma trận hành vi: Từ Giao dịch lần 1 sang Giao dịch lần 2"
)

fig_heatmap.show()

In [17]:
df_success = df_all.copy()

# 1. Lấy sub_cat của giao dịch lần 1 và đổi tên NGAY LẬP TỨC
trans_1_sub = (
    df_success[df_success['trans_rank'] == 1][['userID', 'report_cat', 'report_sub_cat']]
    .rename(columns={'report_cat': 'cat_1', 'report_sub_cat': 'sub_cat_1'})
)

# Lấy sub_cat của giao dịch lần 2 và đổi tên NGAY LẬP TỨC
trans_2_sub = (
    df_success[df_success['trans_rank'] == 2][['userID', 'report_cat', 'report_sub_cat']]
    .rename(columns={'report_cat': 'cat_2', 'report_sub_cat': 'sub_cat_2'})
)

# 2. Gộp 2 lần giao dịch lại để tạo đường dẫn (Behavioral Path)
path_sub_df = trans_1_sub.merge(trans_2_sub, on='userID', how='inner')

# 3. LỌC "SIÊU DANH MỤC": Lúc này cat_1 và cat_2 đã sẵn sàng để gọi
super_flow = path_sub_df[(path_sub_df['cat_1'] == 'Access_Service') & (path_sub_df['cat_2'] == 'Obligation_Payment')]

# 4. Đếm số lượng chuyển đổi giữa các cặp sub_cat
sub_heatmap_data = super_flow.groupby(['sub_cat_1', 'sub_cat_2']).size().reset_index(name='Count')

# 5. Chuyển thành Pivot Table (Ma trận)
sub_heatmap_pivot = sub_heatmap_data.pivot(index='sub_cat_1', columns='sub_cat_2', values='Count').fillna(0)

# 6. Vẽ Sub-Heatmap ngách
fig_sub_heatmap = px.imshow(
    sub_heatmap_pivot,
    labels=dict(x="Lần 2: Sub-cat của Obligation Payment", y="Lần 1: Sub-cat của Dịch vụ (Access)", color="Lượng User"),
    x=sub_heatmap_pivot.columns,
    y=sub_heatmap_pivot.index,
    text_auto=True,
    aspect="auto",
    color_continuous_scale='Purples', 
    title="Đào sâu luồng chuyển đổi ngách: Từ Access_Service sang Obligation_Payment"
)

fig_sub_heatmap.update_layout(xaxis_tickangle=-45)
fig_sub_heatmap.show()

Dựa vào heatmap -> xem luồn chuyển nhiều nhất là `Access_Service` sang `Obligation_payment`

**Bản chất hành vi**: Insight: Người dùng thường sẽ nhận được những khuyến mãi về nạp thẻ điện thoại -> nạp thẻ xong sẽ check những voucher khác hoặc vô tình đc noti là đóng tiền hoá đơn nên sẽ thực hiện `Ulitlity_service`

In [18]:
super_flow = path_sub_df[(path_sub_df['cat_1'] == 'Access_Service') & (path_sub_df['cat_2'] == 'Goods_Transaction')]

# 4. Đếm số lượng chuyển đổi giữa các cặp sub_cat
sub_heatmap_data = super_flow.groupby(['sub_cat_1', 'sub_cat_2']).size().reset_index(name='Count')

# 5. Chuyển thành Pivot Table (Ma trận)
sub_heatmap_pivot = sub_heatmap_data.pivot(index='sub_cat_1', columns='sub_cat_2', values='Count').fillna(0)

# 6. Vẽ Sub-Heatmap ngách
fig_sub_heatmap = px.imshow(
    sub_heatmap_pivot,
    labels=dict(x="Lần 2: Sub-cat của Goods Transaction", y="Lần 1: Sub-cat của Dịch vụ (Access)", color="Lượng User"),
    x=sub_heatmap_pivot.columns,
    y=sub_heatmap_pivot.index,
    text_auto=True,
    aspect="auto",
    color_continuous_scale='Purples', 
    title="Đào sâu luồng chuyển đổi ngách: Từ Access_Service sang Goods_Transaction"
)

fig_sub_heatmap.update_layout(xaxis_tickangle=-45)
fig_sub_heatmap.show()

In [19]:
super_flow = path_sub_df[(path_sub_df['cat_1'] == 'Goods_Transaction') & (path_sub_df['cat_2'] == 'Obligation_Payment')]

# 4. Đếm số lượng chuyển đổi giữa các cặp sub_cat
sub_heatmap_data = super_flow.groupby(['sub_cat_1', 'sub_cat_2']).size().reset_index(name='Count')

# 5. Chuyển thành Pivot Table (Ma trận)
sub_heatmap_pivot = sub_heatmap_data.pivot(index='sub_cat_1', columns='sub_cat_2', values='Count').fillna(0)

# 6. Vẽ Sub-Heatmap ngách
fig_sub_heatmap = px.imshow(
    sub_heatmap_pivot,
    labels=dict(x="Lần 2: Sub-cat của Obligation Payment", y="Lần 1: Sub-cat của Dịch vụ (Access)", color="Lượng User"),
    x=sub_heatmap_pivot.columns,
    y=sub_heatmap_pivot.index,
    text_auto=True,
    aspect="auto",
    color_continuous_scale='Purples', 
    title="Đào sâu luồng chuyển đổi ngách: Từ Access_Service sang Obligation_Payment"
)

fig_sub_heatmap.update_layout(xaxis_tickangle=-45)
fig_sub_heatmap.show()

Đây là hiệu ứng Mồi câu chéo:
- Bản chất: Luồng này rất có thể là kết quả thành công rực rỡ của một chiến dịch thiết kế mồi câu chéo giữa team E-commerce và team Hóa đơn.

- Kịch bản thực tế: Hệ thống tung ra chương trình: "Chốt đơn trên Platform X (hóa đơn > 300k), tặng ngay Voucher giảm 50k khi thanh toán tiền Điện/Nước".

- User mua sắm xong $\rightarrow$ Nhận được Voucher Hóa đơn vào ví $\rightarrow$ Sợ hết hạn nên bấm vào thanh toán luôn tiền điện tháng này.

- Đánh giá Campaign: Nếu luồng này được dẫn dắt bởi Voucher, đây chính là Chiến dịch Cross-sell mẫu mực nhất. Nó dùng một dịch vụ sinh lời/hấp dẫn (Mua sắm) để tạo động lực giải quyết một dịch vụ khô khan (Đóng hóa đơn), giữ toàn bộ dòng tiền của user ở lại trong hệ sinh thái của ứng dụng.

## Phân tích tỉ lệ drop_off khi cross_selling

In [20]:
# 1. Lấy giao dịch đầu tiên của mỗi user (dựa trên df_success đã tạo ở phần trước)
first_trans = df_all[df_all['trans_rank'] == 1][['userID', 'report_cat']]
first_trans.rename(columns={'report_cat': 'first_category'}, inplace=True)

# 2. Lấy các giao dịch từ lần 2 trở đi
subsequent_trans = df_all[df_all['trans_rank'] > 1][['userID', 'report_cat']]

# 3. Tìm những user CÓ dùng dịch vụ KHÁC với dịch vụ đầu tiên
# Merge để đối chiếu danh mục giao dịch sau với danh mục giao dịch đầu
merged_trans = subsequent_trans.merge(first_trans, on='userID')
cross_sell_users = merged_trans[merged_trans['report_cat'] != merged_trans['first_category']]['userID'].unique()

# 4. Gắn cờ (flag) cho nhóm first_trans: 1 = có cross-sell, 0 = ngõ cụt
first_trans['is_cross_sell'] = first_trans['userID'].apply(lambda x: 1 if x in cross_sell_users else 0)

# 5. Gom nhóm theo Dịch vụ đầu tiên để tính toán
gateway_analysis = first_trans.groupby('first_category').agg(
    total_users=('userID', 'count'),          # Lượng traffic đổ vào
    cross_sellers=('is_cross_sell', 'sum')    # Số user đi tiếp sang dịch vụ khác
).reset_index()

# 6. Tính tỷ lệ Cross-sell và Drop-off (Ngõ cụt)
gateway_analysis['cross_sell_rate'] = (gateway_analysis['cross_sellers'] / gateway_analysis['total_users']) * 100
gateway_analysis['drop_off_rate'] = 100 - gateway_analysis['cross_sell_rate']



In [21]:
gateway_analysis.sort_values('cross_sell_rate', ascending=False)

,first_category,total_users,cross_sellers,cross_sell_rate,drop_off_rate
5,Interactive_Service,1742,1105,63.432836,36.567164
6,Mobility_Service,77,48,62.337662,37.662338
2,Financial_Service,73,44,60.273973,39.726027
8,OnDemand_Service,1084,621,57.287823,42.712177
7,Obligation_Payment,11840,6652,56.182432,43.817568
3,Goods_Transaction,2516,1402,55.723370,44.276630
1,Daily_Consumption,408,180,44.117647,55.882353
0,Access_Service,15136,6279,41.483879,58.516121
4,Infrastructure_Service,2,0,0.000000,100.000000


### Giải mã các "Ngõ cụt" (Dead-end) có Drop-off cao

In [22]:
# 1. Đếm tổng số giao dịch thành công của mỗi user
user_trans_counts = df_success['userID'].value_counts().reset_index()
user_trans_counts.columns = ['userID', 'total_transactions']

# 2. Tìm ra danh sách các "User Ngõ cụt" (Chỉ có đúng 1 giao dịch rồi Churn)
dead_end_users = user_trans_counts[user_trans_counts['total_transactions'] == 1]['userID']

# 3. Lọc ra các giao dịch đầu tiên (trans_rank = 1) của TẤT CẢ user để làm gốc so sánh
df_success['trans_rank'] = df_success.groupby('userID').cumcount() + 1
df_first_trans = df_success[df_success['trans_rank'] == 1].copy()

# 4. Gắn cờ (Flag) cho những giao dịch đầu tiên: 1 nếu user đó Churn luôn, 0 nếu user đó đi tiếp
df_first_trans['is_drop_off'] = df_first_trans['userID'].isin(dead_end_users).astype(int)

# 1. Thống kê theo Danh mục, Chiến dịch và Loại khuyến mãi
campaign_drop_analysis = df_first_trans.groupby(['report_cat', 'campaignID', 'promotion_type']).agg(
    total_npu=('userID', 'count'),          # Tổng lượng NPU đổ vào cặp này
    drop_off_users=('is_drop_off', 'sum')   # Số user dùng xong 1 lần rồi bỏ
).reset_index()

# 2. Tính tỷ lệ Drop-off cho từng Campaign bên trong Danh mục
campaign_drop_analysis['campaign_drop_off_rate (%)'] = (campaign_drop_analysis['drop_off_users'] / campaign_drop_analysis['total_npu']) * 100

# 3. Lọc ra các Campaign có lượng user đủ lớn (ví dụ > 100 user) để tránh nhiễu số liệu nhỏ
campaign_drop_analysis = campaign_drop_analysis[campaign_drop_analysis['total_npu'] > 100]

# Sắp xếp xem Campaign nào ở danh mục nào có tỷ lệ "ngõ cụt" nghiêm trọng nhất
bad_campaigns = campaign_drop_analysis.sort_values(by='campaign_drop_off_rate (%)', ascending=False)
bad_campaigns.head(10)

,report_cat,campaignID,promotion_type,total_npu,drop_off_users,campaign_drop_off_rate (%)
352,Obligation_Payment,10632,voucher,182,127,69.780220
102,Daily_Consumption,9179,direct discount,214,112,52.336449
249,Goods_Transaction,9632,voucher,212,99,46.698113
346,Obligation_Payment,10249,voucher,1501,634,42.238508
44,Access_Service,8932,voucher,3745,1482,39.572764
45,Access_Service,8933,voucher,136,47,34.558824
208,Goods_Transaction,8898,voucher,170,55,32.352941
343,Obligation_Payment,10195,voucher,265,85,32.075472
53,Access_Service,9354,direct discount,117,36,30.769231
403,Obligation_Payment,9037,voucher,682,200,29.325513


In [23]:
# Vẽ biểu đồ Sunburst để bóc tách nguyên nhân gây sập phễu
fig_root_cause = px.sunburst(
    bad_campaigns.head(30), # Lấy top 30 cụm có tỷ lệ drop-off tệ nhất hoặc quy mô lớn nhất
    path=['report_cat', 'promotion_type', 'campaignID'], # Hệ phân cấp đào sâu
    values='drop_off_users', # Độ to của ô dựa trên số lượng user bị mất đi
    color='campaign_drop_off_rate (%)',
    color_continuous_scale='OrRd', # Màu cam sang đỏ để cảnh báo
    title='Phân tích Nguyên nhân gốc rễ: Các Campaign gây sập phễu (Drop-off cao)',
    labels={'drop_off_users': 'Số lượng User rời bỏ', 'campaign_drop_off_rate (%)': 'Tỷ lệ Churn ngay sau lần 1 (%)'}
)

fig_root_cause.update_layout(height=600)
fig_root_cause.show()

- Hệ thống đang sử dụng `Access_Service` làm Cửa ngõ (Gateway) để thu hút NPU bằng hình thức Voucher. Tuy nhiên, đây là một thiết kế phễu sai lầm.
Đặc thù của `Access_Service` là tính tức thời, dùng xong là hết nhu cầu, dẫn đến tỷ lệ Drop-off cao. Chúng ta đang tốn chi phí Marketing (CAC) rất cao ở phần dịch vụ này.

Đề xuất: Phải biến `Access_Service` thành bàn đạp cross-sell. Khi NPU dùng Voucher nạp tiền điện thoại lần đầu thành công, màn hình hoàn tất (Success Screen) tuyệt đối không được để trống, mà phải popup ngay lập tức một Voucher Mua sắm (Goods_Transaction) hoặc Thanh toán hóa đơn có thời hạn ngắn (24h) để ép họ chuyển dòng tiền sang danh mục mang lại LTV cao hơn."

## PHân phối time-to-cross-sell

In [24]:
df_success = df_all.copy()

time_1 = df_success[df_success['trans_rank'] == 1][['userID', 'report_cat', 'reqDate']]
time_1.rename(columns={'report_cat': 'cat_1', 'reqDate': 'time_1'}, inplace=True) # Tạo bản copy an toàn

time_2 = df_success[df_success['trans_rank'] == 2][['userID', 'report_cat', 'reqDate']]
time_2.rename(columns={'report_cat': 'cat_2', 'reqDate': 'time_2'}, inplace=True)

# 2. Gộp lại theo userID
df_time = time_1.merge(time_2, on='userID', how='inner')

# 3. Tính khoảng cách thời gian (Time Delta)
# Tính bằng Giờ (Hours) sẽ chi tiết hơn cho các luồng ngắn hạn
df_time['hours_to_cross_sell'] = (df_time['time_2'] - df_time['time_1']).dt.total_seconds() / 3600

# Tính bằng Ngày (Days) để nhìn tổng quan
df_time['days_to_cross_sell'] = df_time['hours_to_cross_sell'] / 24

# 4. Vẽ biểu đồ Phân phối (Histogram) để xem user tập trung cross-sell ở khung thời gian nào
df_time_48h = df_time[df_time['hours_to_cross_sell'] <= 48].copy()

# 2. Vẽ lại biểu đồ Histogram
fig_time_48h = px.histogram(
    df_time_48h, 
    x="hours_to_cross_sell", 
    color="cat_1", 
    nbins=48, # Chia thành 48 cột, mỗi cột đúng 1 giờ
    title="Phóng to: Phân phối Thời gian trễ (Time-to-Cross-sell) trong 48 giờ đầu tiên",
    labels={'hours_to_cross_sell': 'Số giờ chờ (0 - 48h)', 'cat_1': 'Danh mục xuất phát'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# 3. Tối ưu giao diện (chia vạch trục X mỗi 2 giờ cho dễ nhìn)
fig_time_48h.update_xaxes(dtick=2)
fig_time_48h.update_layout(bargap=0.1) # Thêm khoảng cách giữa các cột

fig_time_48h.show()

**Insight**: Gần như 95% hành vi Bán chéo (Cross-sell) diễn ra ngay trong cùng một phiên đăng nhập. Khách hàng nạp tiền, đóng tiền điện, hoặc mua sắm... và họ tiện tay xử lý luôn giao dịch thứ 2 khi dòng tiền và cảm xúc đang liền mạch. Một khi họ đã bấm nút "Home" để thoát app, tỷ lệ họ chủ động quay lại để mua thêm gần như bằng 0.

## NHận xét

### 1. Nhận diện Không gian chuyển đổi: "Ảo giác dữ liệu" vs. "Mỏ vàng thực sự"
Thông qua biểu đồ Heatmap, hệ thống ghi nhận hai luồng hành vi đối lập:
- Ảo giác Bán chéo (Luồng Nạp tiền $\rightarrow$ Thanh toán hóa đơn/Nạp thẻ): Chiếm Volume khổng lồ nhưng thực chất đây chỉ là một Đường dẫn phụ thuộc (Dependent Path). Người dùng nạp tiền vào ví chỉ để làm bước đệm cho đúng 1 nhu cầu duy nhất. Chiến dịch kéo traffic vào luồng này không tạo ra giá trị mở rộng.
- Mỏ vàng Cross-sell (Luồng Mua sắm $\rightarrow$ Thanh toán hóa đơn): Nhóm này lột tả "Hiệu ứng ngày nhận lương" (Payday) và sức mạnh của Voucher chéo. Người dùng chịu chi tiêu, có LTV (Giá trị vòng đời) cao và coi ứng dụng là một trung tâm tiện ích thực thụ.

### 2. Chẩn đoán sập phễu: Bắt bệnh rò rỉ tại Access Service
Kết hợp biểu đồ tỷ lệ Drop-off và Sunburst, nguyên nhân gốc rễ gây thủng phễu đã lộ diện:

= Mảng tối của Voucher: Access_Service có tỷ lệ rớt đài lên tới 90% sau lần chạm đầu tiên.

- Thủ phạm: Việc lạm dụng cơ chế Giảm thẳng tiền mặt (Direct Discount) đã biến ứng dụng thành mỏ ngầm cho các "thợ săn mã" (Promo Hunters). Họ hoàn thành một giao dịch duy nhất để lấy quả ngọt rồi lập tức rời bỏ nền tảng.

### 3. Yếu tố Thời gian: Định lý "Ngay bây giờ hoặc Không bao giờ"
Phân phối Time-to-Cross-sell trong 48 giờ vạch trần thói quen tiêu dùng cực đoan của NPU:

- Giao dịch một chạm (0 - 1 giờ): 95% hành vi bán chéo diễn ra tức thì trong cùng một phiên đăng nhập. Nếu người dùng thoát app, cơ hội chốt sale gần như bằng 0.

## Phân tích Sự đa dạng của Nguồn tiền (Source of Fund - SoF)

In [25]:
user_sof = df_success.groupby('userID')['sof'].nunique().reset_index(name='unique_sof_count')

# Phân nhóm User theo số lượng nguồn tiền
user_sof['sof_segment'] = user_sof['unique_sof_count'].apply(
    lambda x: '1 Nguồn tiền (Rủi ro Churn)' if x == 1 
    else ('2 Nguồn tiền (Tin tưởng)' if x == 2 else '3+ Nguồn tiền (Khách ruột)')
)


# ==========================================
# BƯỚC 2: GẮN CAMPAIGN ĐẦU TIÊN (FIRST-TOUCH)
# ==========================================
# Lấy Campaign đầu tiên của mỗi User (Đã lọc bỏ Campaign = 0)
df_real_camp = df_all[df_all['campaignID'] != 0].dropna(subset=['campaignID'])
first_touch = df_real_camp.groupby('userID')['campaignID'].first().reset_index()
first_touch.rename(columns={'campaignID': 'acquisition_campaign'}, inplace=True)

# Merge thông tin Campaign vào bảng đếm SoF
user_sof_campaign = user_sof.merge(first_touch, on='userID', how='left')

# Điền 'Organic (0)' cho những user không có Campaign nào
user_sof_campaign['acquisition_campaign'] = user_sof_campaign['acquisition_campaign'].fillna('0')


# ==========================================
# BƯỚC 3: LỌC TOP 10 CAMPAIGN & TÍNH TỶ LỆ %
# ==========================================
# Đếm số lượng User theo từng Campaign và từng Segment SoF
sof_distribution = user_sof_campaign.groupby(['acquisition_campaign', 'sof_segment']).size().reset_index(name='user_count')

# Tìm Top 10 Campaign mang về nhiều User nhất (không tính Organic)
campaign_volume = sof_distribution[sof_distribution['acquisition_campaign'] != '0'].groupby('acquisition_campaign')['user_count'].sum().reset_index()
top_10_campaigns = campaign_volume.sort_values('user_count', ascending=False).head(10)['acquisition_campaign'].tolist()

# Gom nhóm các Campaign nhỏ thành 'Others'
sof_distribution['campaign_grouped'] = sof_distribution['acquisition_campaign'].apply(
    lambda x: 'Organic (0)' if x == '0' else (str(int(float(x))) if x in top_10_campaigns else 'Others')
)

# Tính lại tổng sau khi gom nhóm
final_sof_grouped = sof_distribution.groupby(['campaign_grouped', 'sof_segment'])['user_count'].sum().reset_index()

# Tính tỷ lệ % user của từng phân khúc SoF trong mỗi Campaign
final_sof_grouped['pct_users'] = final_sof_grouped.groupby('campaign_grouped')['user_count'].transform(lambda x: (x / x.sum()) * 100)


# ==========================================
# BƯỚC 4: VẼ BIỂU ĐỒ 100% STACKED BAR CHART
# ==========================================
# Sắp xếp trục X: Top 10 Campaign -> Organic -> Others
order_x = [str(int(float(c))) for c in top_10_campaigns] + ['Organic (0)', 'Others']
# Sắp xếp thứ tự chú thích (Legend)
order_segment = ['1 Nguồn tiền (Rủi ro Churn)', '2 Nguồn tiền (Tin tưởng)', '3+ Nguồn tiền (Khách ruột)']

fig_sof = px.bar(
    final_sof_grouped, 
    x='campaign_grouped', 
    y='pct_users', 
    color='sof_segment', 
    title='Đánh giá mức độ Tin tưởng: Phân phối Đa dạng Nguồn tiền (SoF) theo Campaign',
    labels={
        'campaign_grouped': 'Chiến dịch (Acquisition Campaign)', 
        'pct_users': 'Tỷ lệ User (%)', 
        'sof_segment': 'Phân khúc Nguồn tiền'
    },
    text_auto='.1f',
    category_orders={
        "campaign_grouped": order_x,
        "sof_segment": order_segment
    },
    # Dùng dải màu sắc thái nóng/lạnh để lột tả mức độ tốt/xấu (Xanh đậm là tốt, Đỏ/Cam là rủi ro)
    color_discrete_map={
        '1 Nguồn tiền (Rủi ro Churn)': '#e07a5f',  # Màu cam gạch (Cảnh báo)
        '2 Nguồn tiền (Tin tưởng)': '#81b29a',     # Màu xanh lá mạ (Tốt)
        '3+ Nguồn tiền (Khách ruột)': '#3d405b'    # Màu xanh than (Rất tốt)
    }
)

fig_sof.update_layout(
    xaxis_tickangle=-45, 
    yaxis_title="Tỷ lệ NPU (%)",
    legend_title="Sự đa dạng Nguồn tiền",
    barmode='stack'
)

fig_sof.show()

In [26]:
df_success = df_all.copy()

user_stats = df_success.groupby('userID').agg(
    unique_sof=('sof', 'nunique'),
    total_trans=('trans_rank', 'max') 
).reset_index()

# 2. Phân loại Segment Nguồn tiền
user_stats['sof_segment'] = user_stats['unique_sof'].apply(
    lambda x: '1 Nguồn tiền' if x == 1 else ('2 Nguồn tiền' if x == 2 else '3+ Nguồn tiền')
)

# 3. Phân loại Trạng thái: Rời bỏ (Drop-off) vs. Ở lại (Retained)
user_stats['status'] = user_stats['total_trans'].apply(
    lambda x: 'Drop-off (Chỉ 1 giao dịch)' if x == 1 else 'Retained (2+ giao dịch)'
)

# 4. Gom nhóm tính Tỷ lệ % để vẽ biểu đồ
status_dist = user_stats.groupby(['sof_segment', 'status']).size().reset_index(name='user_count')
status_dist['pct_users'] = status_dist.groupby('sof_segment')['user_count'].transform(lambda x: (x / x.sum()) * 100)

# 5. Vẽ biểu đồ Bar Chart (100% Stacked)
fig_retention = px.bar(
    status_dist, 
    x='sof_segment', 
    y='pct_users', 
    color='status',
    title='Bằng chứng thép: Tỷ lệ Rời bỏ (Drop-off) theo Độ đa dạng Nguồn tiền',
    labels={
        'sof_segment': 'Độ đa dạng Nguồn tiền', 
        'pct_users': 'Tỷ lệ User (%)', 
        'status': 'Trạng thái'
    },
    text_auto='.1f',
    category_orders={"sof_segment": ['1 Nguồn tiền', '2 Nguồn tiền', '3+ Nguồn tiền']},
    # Màu sắc nổi bật sự tương phản
    color_discrete_map={
        'Drop-off (Chỉ 1 giao dịch)': '#d62828', # Đỏ cảnh báo
        'Retained (2+ giao dịch)': '#2a9d8f'     # Xanh an toàn
    }
)

fig_retention.update_layout(barmode='stack', yaxis_title="Tỷ lệ %")
fig_retention.show()

In [27]:
# ==========================================
# BƯỚC 1: TÍNH TOÁN CÁC CHỈ SỐ CHO TỪNG USER
# ==========================================
# Gom nhóm theo userID và tính toán 4 chỉ số cốt lõi
user_metrics = df_success.groupby('userID').agg(
    unique_sof=('sof', 'nunique'),          # Số lượng Nguồn tiền
    total_trans=('userID', 'count'),           # Tần suất (Số lượng giao dịch)
    total_spend=('userChargeAmount', 'sum'),             # Tổng chi tiêu (LTV/ARPU) - Thay 'amount' bằng tên cột tiền của bạn
    unique_categories=('report_cat', 'nunique') # Số lượng Danh mục (Độ phủ hệ sinh thái)
).reset_index()


# ==========================================
# BƯỚC 2: PHÂN LOẠI PHÂN KHÚC NGUỒN TIỀN
# ==========================================
user_metrics['sof_segment'] = user_metrics['unique_sof'].apply(
    lambda x: '1 Nguồn tiền' if x == 1 
    else ('2 Nguồn tiền' if x == 2 else '3+ Nguồn tiền')
)


# ==========================================
# BƯỚC 3: TỔNG HỢP & TÍNH TRUNG BÌNH THEO PHÂN KHÚC
# ==========================================
# Gom nhóm theo segment và tính giá trị trung bình (mean)
loyalty_summary = user_metrics.groupby('sof_segment').agg(
    user_count=('userID', 'count'),
    avg_transactions=('total_trans', 'mean'),
    avg_spend=('total_spend', 'mean'),
    avg_categories=('unique_categories', 'mean')
).reset_index()

# Tính thêm tỷ lệ % User để sếp nắm được dung lượng của từng tập
total_users = loyalty_summary['user_count'].sum()
loyalty_summary['pct_users'] = (loyalty_summary['user_count'] / total_users * 100)

# ==========================================
# BƯỚC 4: FORMAT BẢNG ĐỂ TRÌNH BÀY CHO ĐẸP
# ==========================================
# Sắp xếp lại thứ tự cột cho logic
loyalty_summary = loyalty_summary[[
    'sof_segment', 'user_count', 'pct_users', 
    'avg_transactions', 'avg_spend', 'avg_categories'
]]

# Format các con số cho dễ đọc (làm tròn)
loyalty_summary['pct_users'] = loyalty_summary['pct_users'].round(1).astype(str) + '%'
loyalty_summary['avg_transactions'] = loyalty_summary['avg_transactions'].round(1)
loyalty_summary['avg_spend'] = loyalty_summary['avg_spend'].map('{:,.0f}'.format) # Format tiền tệ có dấu phẩy
loyalty_summary['avg_categories'] = loyalty_summary['avg_categories'].round(1)

# Đổi tên cột sang tiếng Việt chuẩn Business
loyalty_summary.columns = [
    'Phân khúc SoF', 'Số lượng User', 'Tỷ trọng (%)', 
    'Giao dịch TB / User', 'Chi tiêu TB / User (ARPU)', 'Danh mục TB / User'
]

# In bảng kết quả ra màn hình
print("BẢNG ĐÁNH GIÁ CHẤT LƯỢNG NPU VÀ LÒNG TRUNG THÀNH QUA NGUỒN TIỀN:")
display(loyalty_summary)

BẢNG ĐÁNH GIÁ CHẤT LƯỢNG NPU VÀ LÒNG TRUNG THÀNH QUA NGUỒN TIỀN:


,Phân khúc SoF,Số lượng User,Tỷ trọng (%),Giao dịch TB / User,Chi tiêu TB / User (ARPU),Danh mục TB / User
0,1 Nguồn tiền,25264,76.8%,6.2,"1,258,925",1.6
1,2 Nguồn tiền,7488,22.8%,14.5,"2,713,235",2.3
2,3+ Nguồn tiền,126,0.4%,23.0,"8,610,509",2.6


### Insight 1: Hiệu ứng "Cấp số nhân" khi vượt rào cản thứ 2 (The Multiplier Effect)

### Insight 2: Bẫy "Dung lượng ảo" của hệ thống (The Volume Trap)
Nhìn vào cột tỷ trọng, chúng ta thấy một sự thật khá phũ phàng: 76.7% (gần 26,000 users) đang nằm ở đáy của tháp giá trị. * Mặc dù họ chiếm số đông tuyệt đối, nhưng sức mua của họ rất yếu và độ phủ danh mục mỏng (chỉ 1.6).

- Hành động: Bạn dùng số liệu này để phản biện lại cách chạy chiến dịch Marketing cũ. Thay vì đổ tiền mang về thêm 10,000 user mới (nhưng 76% lại rơi vào nhóm 1 Nguồn tiền), công ty sẽ lãi to nếu dùng số tiền đó làm Gamification để chuyển đổi (upsell) 25,000 user hiện tại từ mốc 1 SoF lên mốc 2 SoF.

### Insight 3: Chân dung "Tệp Siêu VIP" (The Whales)
Dù nhóm 3+ Nguồn tiền chỉ chiếm một "hạt cát" 0.4% (128 users), nhưng sức mạnh dòng tiền của họ vô cùng khủng khiếp:

- Chi tiêu trung bình lên tới 8.6 triệu VNĐ (gấp gần 7 lần nhóm đáy).

- Họ thực hiện gần 23 giao dịch và càn quét gần 3 danh mục dịch vụ.

- Đề xuất chiến lược: Đây là tệp "Đại sứ thương hiệu". Team CRM không được gửi cho nhóm này những tin nhắn rác kiểu "Tặng voucher 10k". Họ cần một Đặc quyền VIP riêng biệt (ví dụ: CSKH 1-1 không cần chờ tổng đài, quà sinh nhật hiện vật) để giữ chân dòng tiền khổng lồ này.

## NHẬN XÉT

### Đối với Tệp khách hàng: Đa dạng Nguồn tiền = Lòng trung thành tuyệt đối
- Chỉ báo sớm (Leading Indicator) của Retention: Lòng trung thành không tự nhiên sinh ra, nó bắt nguồn từ Chi phí chuyển đổi (Switching Cost). Khi một người dùng chỉ dùng ví điện tử nạp sẵn (1 SoF), họ có thể xóa app không thương tiếc ngay khi xài hết tiền ảo hoặc hết voucher.

- Sự tin tưởng (Trust) & Ràng buộc (Lock-in): Nhưng khi họ đã cất công nhập số thẻ tín dụng, xác thực OTP, liên kết tài khoản ngân hàng chính (2+ SoF)... họ đã giao phó sự bảo mật tài chính cho ứng dụng. Tâm lý "đã mất công cài đặt rồi thì dùng luôn cho tiện" khiến họ dính chặt (stickiness) với hệ sinh thái. Số liệu 0% Drop-off và chi tiêu gấp đôi đã chứng minh điều đó.

### Đối với Campaign:
- Kẻ thù của "Chỉ số phù phiếm" (Vanity Metrics): Trước đây, một Campaign mang về 10,000 users với giá CPA (Chi phí/User) cực rẻ có thể được tung hô là thành công. Nhưng nếu 95% user đó chỉ dùng 1 nguồn tiền, thì chiến dịch đó thực chất là một "Hố đen đốt tiền", chỉ mang về thợ săn mã khuyến mãi.

- Định nghĩa lại Campaign thành công: Một chiến dịch thực sự thành công không nằm ở việc nó mang lại bao nhiêu user tải app, mà nằm ở tỷ lệ % user nó mang về chịu kích hoạt nguồn tiền thứ 2. Campaign 7421 chính là ví dụ mẫu mực cho việc kéo đúng tệp khách hàng "có chất lượng, có sức mua thực sự".